# Astropilote — Exploratory Data Analysis of the corpus

**Goal of this notebook.** Before trusting a RAG system, we look at the data it retrieves from. Here we answer seven simple questions about the corpus (`data/processed/documents.jsonl` and `chunks.jsonl`):

1. What kinds of objects does it contain?
2. Which constellations are well covered, and which are not?
3. How bright are the objects (magnitudes)?
4. Where is information missing?
5. How long are the texts (documents and chunks)?
6. How much of it can actually be observed from Paris?
7. How much of the corpus do the evaluation questions actually test?

Each section ends with a **Takeaway** explaining why it matters for the RAG, the agent or the test questions (`eval/golden_charles.jsonl`).
It complements `corpus_and_rag_visualizations.ipynb` (sky map, magnitude vs size, ablations), which it does not repeat.

*Only `numpy` and `matplotlib` are needed (both are installed with `uv sync --extra dev`).*

In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('..').resolve()   # the notebook lives in notebooks/, the data in data/
docs = [json.loads(l) for l in (ROOT / 'data/processed/documents.jsonl').read_text().splitlines()]
chunks = [json.loads(l) for l in (ROOT / 'data/processed/chunks.jsonl').read_text().splitlines()]

# A "document" = one object (or constellation). A "chunk" = a short passage of a document, the unit that gets retrieved.
# 'Constellation' documents describe a whole constellation; the others describe a single deep-sky object.
objects = [d for d in docs if d['catalog'] != 'Constellation']
constellation_docs = [d for d in docs if d['catalog'] == 'Constellation']

print(f'{len(docs)} documents ({len(objects)} deep-sky objects + {len(constellation_docs)} constellations)')
print(f'{len(chunks)} chunks')
print('Documents per catalog:', dict(Counter(d['catalog'] for d in docs)))

# Visual theme: same dark look and palette as corpus_and_rag_visualizations.ipynb
BG, PANEL, EDGE, TEXT, MUTED = '#0b1120', '#111827', '#334155', '#f8fafc', '#94a3b8'
SKY, EMERALD, INDIGO, AMBER, PINK, VIOLET, RED = '#38bdf8', '#34d399', '#818cf8', '#fbbf24', '#f472b6', '#a78bfa', '#f87171'
CAT_COLORS = {'Messier': INDIGO, 'Caldwell': EMERALD, 'Herschel': SKY, 'Constellation': AMBER}   # one color per catalog, everywhere

plt.rcParams.update({
    'figure.dpi': 120, 'figure.facecolor': BG, 'savefig.facecolor': BG,
    'axes.facecolor': PANEL, 'axes.edgecolor': EDGE, 'axes.labelcolor': MUTED,
    'axes.titlecolor': TEXT, 'axes.titleweight': 'bold', 'axes.titlesize': 13, 'axes.titlepad': 12,
    'text.color': TEXT, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.grid': True, 'grid.color': EDGE, 'grid.linestyle': '--', 'grid.alpha': 0.45, 'axes.axisbelow': True,
    'legend.facecolor': PANEL, 'legend.edgecolor': EDGE, 'legend.labelcolor': TEXT,
})

## 1. Object types

Each deep-sky object has an `object_type` (open cluster, galaxy, ...) taken from OpenNGC. The left chart counts them; the right one shows which catalog each type comes from.

In [ ]:
type_counts = Counter(d['metadata']['object_type'] for d in objects)
types = [t for t, _ in type_counts.most_common()]
catalogs = ['Messier', 'Caldwell', 'Herschel']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1, 1.2]})

# Left: total per type
vals = [type_counts[t] for t in types]
ax1.barh(types[::-1], vals[::-1], color=SKY)
for y, v in enumerate(vals[::-1]):
    ax1.text(v + 2, y, str(v), va='center', fontsize=9)
ax1.set_title(f'Deep-sky objects by type (n={len(objects)})')
ax1.set_xlim(0, 225)
ax1.set_xlabel('number of documents')

# Right: stacked by catalog
left = np.zeros(len(types))
for cat, color in [(c, CAT_COLORS[c]) for c in catalogs]:
    row = np.array([sum(1 for d in objects if d['catalog'] == cat and d['metadata']['object_type'] == t) for t in types])
    ax2.barh(types[::-1], row[::-1], left=left[::-1], label=cat, color=color)
    left += row
ax2.set_title('Same, split by catalog')
ax2.set_xlabel('number of documents')
ax2.legend()
plt.tight_layout(); plt.show()

share_oc = type_counts['open_cluster'] / len(objects)
print(f"Open clusters: {type_counts['open_cluster']} ({share_oc:.0%}); galaxies: {type_counts['galaxy']}; globulars: {type_counts['globular_cluster']}")
print('Rare types (<= 4 documents):', {t: n for t, n in type_counts.items() if n <= 4})

**Takeaway.** The corpus is dominated by open clusters (201 of 418, 48%), then galaxies (76) and globular clusters (54). Types such as supernova remnants or reflection nebulae have only 4 documents each, double stars 3. The retriever therefore has far fewer examples to find for those, and a question on them is a harder test (see `q058`, the Jewel Bug Nebula, a planetary nebula, and the Crab Nebula `q048`, a supernova remnant). Also note that 151 of the 201 open clusters come from the Herschel list: Herschel adds mostly clusters (and almost no galaxies, only 2), while galaxies come from Messier and Caldwell.

## 2. Constellations

Each deep-sky object belongs to one constellation (IAU 3-letter code). We count how many objects each constellation holds, and check which of the 88 constellations have **no** object at all.

In [ ]:
con_counts = Counter(d['metadata']['constellation'] for d in objects)
top = con_counts.most_common(15)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar([c for c, _ in top], [n for _, n in top], color=SKY)
for i, (_, n) in enumerate(top):
    ax.text(i, n + 0.3, str(n), ha='center', fontsize=9)
ax.set_title('Top 15 constellations by number of deep-sky objects')
ax.set_ylabel('objects')
plt.tight_layout(); plt.show()

all_cons = {d['metadata']['constellation'] for d in constellation_docs}
empty = sorted(all_cons - set(con_counts))
print(f'{len(con_counts)} constellations contain at least one object; {len(empty)} contain none:')
print(', '.join(empty))
print(f"Median objects per constellation (non-empty): {int(np.median(list(con_counts.values())))}")

**Takeaway.** A handful of constellations (Sagittarius, Cygnus, Monoceros, Puppis, Scorpius) hold 19 to 23 objects each, because they lie along the Milky Way, while the typical non-empty constellation holds only 3. Twenty constellations (e.g. Ursa Minor, Aries, Eridanus, Crater) have a constellation document but **no deep-sky object**. A question such as "which objects are in Eridanus?" has no deep-sky answer in the corpus, which is a good edge case for the agent's behaviour. Conversely, queries about dense constellations (Cygnus, Sagittarius) are where retrieval can miss objects, as the existing error analysis found for `q017`.

## 3. Brightness (magnitudes)

The **magnitude** measures how bright an object looks: the *lower* the number, the brighter. Rough rules of thumb: up to about 6 is visible to the naked eye under a dark sky, up to about 8.5 with binoculars, and anything fainter needs a telescope. Only objects with a known magnitude are counted here (the missing ones are studied in section 4).

In [ ]:
NAKED_EYE, BINOCULARS = 6.0, 8.5
mags = {cat: np.array([d['metadata']['magnitude'] for d in objects
                       if d['catalog'] == cat and d['metadata']['magnitude'] is not None]) for cat in catalogs}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={'width_ratios': [1.4, 1]})

# Left: histogram of magnitudes, stacked by catalog
bins = np.arange(0, 13.5, 0.5)
ax1.hist([mags[c] for c in catalogs], bins=bins, stacked=True,
         color=[CAT_COLORS[c] for c in catalogs], label=[f'{c} (n={len(mags[c])})' for c in catalogs])
ax1.axvline(NAKED_EYE, color=EMERALD, ls='--', lw=1.5)
ax1.axvline(BINOCULARS, color=AMBER, ls='--', lw=1.5)
ymax = ax1.get_ylim()[1]
ax1.text(NAKED_EYE - 0.15, ymax * 0.93, 'naked eye', color=EMERALD, ha='right', fontsize=9)
ax1.text(BINOCULARS + 0.15, ymax * 0.93, 'binoculars limit', color=AMBER, ha='left', fontsize=9)
ax1.set_title('Magnitude of the deep-sky objects (lower = brighter)')
ax1.set_xlabel('visual magnitude'); ax1.set_ylabel('objects'); ax1.legend(loc='upper left', bbox_to_anchor=(0.0, 0.85))

# Right: share of each catalog by equipment needed
levels = [('naked eye (<= 6)', lambda m: m <= NAKED_EYE, EMERALD),
          ('binoculars (6 to 8.5)', lambda m: (m > NAKED_EYE) & (m <= BINOCULARS), SKY),
          ('telescope (> 8.5)', lambda m: m > BINOCULARS, VIOLET)]
left = np.zeros(len(catalogs))
for label, test, color in levels:
    share = np.array([test(mags[c]).mean() * 100 for c in catalogs])
    ax2.barh(catalogs[::-1], share[::-1], left=left[::-1], color=color, label=label)
    for i, (sh, lf) in enumerate(zip(share[::-1], left[::-1])):
        if sh > 7:
            ax2.text(lf + sh / 2, i, f'{sh:.0f}%', ha='center', va='center', color=BG, fontsize=9, fontweight='bold')
    left += share
ax2.set_xlim(0, 100); ax2.set_xlabel('% of objects with a known magnitude')
ax2.set_title('Equipment needed, by catalog')
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=1, fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'catalog':10s} {'n':>4s} {'median':>7s} {'<=6':>5s} {'6-8.5':>6s} {'>8.5':>5s}")
for c in catalogs:
    m = mags[c]
    print(f'{c:10s} {len(m):4d} {np.median(m):7.1f} {int((m <= NAKED_EYE).sum()):5d} '
          f'{int(((m > NAKED_EYE) & (m <= BINOCULARS)).sum()):6d} {int((m > BINOCULARS).sum()):5d}')
all_m = np.concatenate(list(mags.values()))
print(f'All: {len(all_m)} objects, faintest = {all_m.max():.1f}, '
      f'{(all_m > BINOCULARS).sum()} ({(all_m > BINOCULARS).mean():.0%}) need a telescope')

**Takeaway.** The typical object is moderately bright (median about 7.4 overall, 7.2 for Herschel and 7.8 for Messier), but the catalogs differ. Caldwell is the faintest (median 9.1): 50 of its 89 objects with a magnitude (56%) are beyond the binocular limit. Herschel is the most uniform: 81% of its objects lie between magnitude 6 and 8.5 and only one is fainter than 8.5, which suggests that this list was selected among the brighter objects. For the agent, this means that **the right target depends on the observer's equipment**: with binoculars or a very small instrument, about a fifth of the objects (22%) are out of reach, and a good answer must use the magnitude and size (the field-of-view tool) rather than just list famous objects.

## 4. Missing data

Some fields can be empty. We check, per catalog, which objects lack a **magnitude** (brightness) or **coordinates**. The 88 constellation documents are excluded: a constellation has no magnitude by nature.

In [ ]:
rows = []
for cat in catalogs:
    sub = [d for d in objects if d['catalog'] == cat]
    no_mag = [d['doc_id'] for d in sub if d['metadata']['magnitude'] is None]
    no_xy = [d['doc_id'] for d in sub if d['metadata']['ra_deg'] is None or d['metadata']['dec_deg'] is None]
    no_size = [d['doc_id'] for d in sub if d['metadata']['size_major_arcmin'] is None]
    rows.append((cat, len(sub), no_mag, no_xy, no_size))

print(f"{'catalog':10s} {'docs':>5s} {'no magnitude':>13s} {'no coords':>10s} {'no size':>8s}")
for cat, n, a, b, c in rows:
    print(f'{cat:10s} {n:5d} {len(a):13d} {len(b):10d} {len(c):8d}')

fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(rows)); w = 0.27
ax.bar(x - w, [len(r[2]) / r[1] * 100 for r in rows], w, label='no magnitude', color=RED)
ax.bar(x, [len(r[4]) / r[1] * 100 for r in rows], w, label='no size', color=AMBER)
ax.bar(x + w, [len(r[3]) / r[1] * 100 for r in rows], w, label='no coordinates', color=VIOLET)
ax.set_xticks(x); ax.set_xticklabels([r[0] for r in rows])
ax.set_ylabel('% of documents'); ax.set_title('Missing values by catalog'); ax.legend()
plt.tight_layout(); plt.show()

print('\nObjects without magnitude:', [d for r in rows for d in r[2]])
print('Objects without coordinates:', [d for r in rows for d in r[3]])

**Takeaway.** 27 of the 418 objects have no magnitude (7 Messier, 20 Caldwell, 0 Herschel). The field is already empty in the **source** data (OpenNGC), it is not an error in our download. Three Caldwell objects (`C009`, `C041`, `C099`) have no coordinates either, so the visibility tool cannot compute anything for them. These gaps justify test questions on a missing magnitude (`q044`, M35): the agent must say the value is not available instead of inventing one.

## 5. Text sizes

The retriever works on **chunks** (short passages). Their length matters: too short and the passage lacks context, too long and the answer is diluted. Sizes below are in **words** (the ingestion log reports about 193 tokens per chunk on average, and a token is a bit smaller than a word).

In [ ]:
words = np.array([len(c['text'].split()) for c in chunks])
chunks_per_doc = np.array([d['num_chunks'] for d in docs])
by_cat_chunks = defaultdict(int)
for c in chunks:
    by_cat_chunks[c['metadata']['catalog']] += 1

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(words, bins=40, color=SKY)
axes[0].axvline(np.median(words), color=TEXT, ls='--', label=f'median = {int(np.median(words))}')
axes[0].set_title('Words per chunk'); axes[0].set_xlabel('words'); axes[0].legend()

axes[1].hist(chunks_per_doc, bins=range(1, 52, 2), color=AMBER)
axes[1].set_yscale('log')
axes[1].set_title('Chunks per document (log scale)'); axes[1].set_xlabel('chunks in one document')

cats = ['Messier', 'Caldwell', 'Herschel', 'Constellation']
n_docs = [sum(1 for d in docs if d['catalog'] == c) for c in cats]
axes[2].bar(cats, [by_cat_chunks[c] / n for c, n in zip(cats, n_docs)], color=[CAT_COLORS[c] for c in cats])
axes[2].set_title('Average chunks per document, by catalog')
axes[2].set_ylabel('chunks / document')
plt.tight_layout(); plt.show()

print(f'Words per chunk: min {words.min()}, 5th pct {int(np.percentile(words, 5))}, median {int(np.median(words))}, 95th pct {int(np.percentile(words, 95))}, max {words.max()}')
print(f'Chunks per document: median {int(np.median(chunks_per_doc))}, max {chunks_per_doc.max()}')
longest = sorted(docs, key=lambda d: -d['num_chunks'])[:5]
print('Documents with the most chunks:', [(d['doc_id'], d['title'], d['num_chunks']) for d in longest])
print('Average chunks per document:', {c: round(by_cat_chunks[c] / n, 1) for c, n in zip(cats, n_docs)})

**Takeaway.** Chunk length has two clusters: a large group of near-maximum chunks (about 200 to 230 words, the size cap of the chunker) and a long tail of shorter ones (short Wikipedia sections), with a median of 155 words and nothing above 234. Documents, however, are very unequal: the median document has 4 chunks but the longest (Auriga) has 49, and the Andromeda Galaxy has 41. Constellations average 13.9 chunks, Messier objects 5.0, Caldwell 3.6 and Herschel only 2.6: famous objects and constellations have long Wikipedia pages and therefore **many chunks**, while obscure Herschel objects often have 1 or 2. A query about a famous object has more chances to match something (and more risk of retrieving several chunks of the same document), whereas an obscure object relies on one or two short chunks. The `[DOC_ID | Title | Type ...]` header prepended to each chunk helps because it keeps the object identity in every passage.

## 6. What can be observed from Paris?

An object can only be observed if it rises high enough. For a site at latitude `lat`, an object at declination `dec` reaches a maximum altitude of

`max_altitude = 90° − |lat − dec|`

This is the theoretical peak, reached when the object crosses the meridian (possibly in daytime). The project's planner is stricter: it only looks at the **dark night window** and uses a **20° minimum altitude floor** (below that, the atmosphere degrades the image). With Paris at about 48.85° N:
- `max_altitude ≥ 0°` (rises at all) needs `dec ≥ about −41°`;
- `max_altitude ≥ 20°` (usable for the planner) needs `dec ≥ about −21°`.

In [ ]:
PARIS_LAT = 48.85
FLOOR = 20.0

with_dec = [d for d in objects if d['metadata']['dec_deg'] is not None]
dec = np.array([d['metadata']['dec_deg'] for d in with_dec])
max_alt = 90 - np.abs(PARIS_LAT - dec)

dec_horizon = PARIS_LAT - 90            # rises at all
dec_floor = PARIS_LAT - (90 - FLOOR)    # reaches the 20 degree floor

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={'width_ratios': [1.4, 1]})

ax1.hist(dec, bins=np.arange(-90, 91, 5), color=SKY)
ax1.axvline(dec_floor, color=RED, ls='--', label=f'20 deg floor (dec = {dec_floor:.1f})')
ax1.axvline(dec_horizon, color=TEXT, ls=':', label=f'never rises (dec < {dec_horizon:.1f})')
ax1.set_title('Declination of the 415 objects with coordinates'); ax1.set_xlabel('declination (deg)'); ax1.set_ylabel('objects')
ax1.legend()

groups = {'usable (max alt >= 20 deg)': max_alt >= FLOOR,
          'too low (0 to 20 deg)': (max_alt >= 0) & (max_alt < FLOOR),
          'never rises (< 0 deg)': max_alt < 0}
counts = [int(m.sum()) for m in groups.values()]
bars = ax2.bar(range(3), counts, color=[EMERALD, AMBER, RED])
ax2.set_xticks(range(3)); ax2.set_xticklabels(['usable', 'too low', 'never rises'])
for b, n in zip(bars, counts):
    ax2.text(b.get_x() + b.get_width() / 2, n + 4, f'{n}\n({n / len(dec):.0%})', ha='center', fontsize=9)
ax2.set_ylim(0, max(counts) * 1.2); ax2.set_title('Observability from Paris'); ax2.set_ylabel('objects')
plt.tight_layout(); plt.show()

for name, m in groups.items():
    print(f'{name:30s}: {int(m.sum())}')

# Which catalogs are the most affected?
print()
for cat in catalogs:
    idx = [i for i, d in enumerate(with_dec) if d['catalog'] == cat]
    usable = int((max_alt[idx] >= FLOOR).sum())
    print(f'{cat:9s}: {usable}/{len(idx)} usable from Paris ({usable / len(idx):.0%})')

**Takeaway.** Only 266 of the 415 objects with coordinates (64%) are usable from Paris: 61 stay below the 20° floor and 88 never rise at all. Messier is the most favourable catalog (81% usable), while Caldwell and Herschel are only 58% usable because they include the deep southern sky (for example Omega Centauri or the Tarantula Nebula). This is exactly why the planner has to **compute** visibility (with a latitude, a date and the 20° floor) instead of letting the language model guess. It motivates the test questions on southern objects: `q041` (Carina Nebula, which never rises from Paris), `q042` (Tarantula Nebula from Marseille, also never visible), `q043` (M7, which peaks around 6° and so stays under the 20° floor) and `q057` (an impossible place, as a trap).

## 7. How much of the corpus do the evaluation questions test?

The retrieval score reported by the project (`retrieval_recall` = 0.99) is measured on the 40 questions of `eval/golden.jsonl`. Each question lists its **gold documents**, the documents that must be found to answer it. To judge how far that score can be trusted, we measure which part of the corpus these gold documents represent, before and after adding our 19 questions (`eval/golden_charles.jsonl`, not yet run through the agent). Coverage here means: *is this document the expected answer of at least one question?*

In [ ]:
def load_jsonl(path):
    return [json.loads(l) for l in path.read_text().splitlines()] if path.exists() else []

golden = load_jsonl(ROOT / 'eval/golden.jsonl')
golden_charles = load_jsonl(ROOT / 'eval/golden_charles.jsonl')      # empty list if the file is not there
results = {r['id']: r for r in load_jsonl(ROOT / 'eval/results.jsonl')}
doc_by_id = {d['doc_id']: d for d in docs}

gold_base = {g for q in golden for g in q['gold_docs']}
gold_all = gold_base | {g for q in golden_charles for g in q['gold_docs']}
n_trap = sum(1 for q in golden + golden_charles if not q['gold_docs'])

# Same definition as eval/run_eval.py: share of the gold documents present in the retrieved ids
n_gold = hits = 0
missed = []
for q in golden:
    for g in q['gold_docs']:
        n_gold += 1
        if g in results[q['id']]['retrieved_ids']:
            hits += 1
        else:
            missed.append((q['id'], g))

def coverage(gold_ids, group_key, groups):
    out = {}
    for grp in groups:
        members = [d['doc_id'] for d in docs if group_key(d) == grp]
        out[grp] = 100 * sum(1 for m in members if m in gold_ids) / len(members)
    return out

cat_groups = ['Messier', 'Caldwell', 'Herschel', 'Constellation']
type_groups = [t for t, _ in type_counts.most_common()]
type_key = lambda d: d['metadata']['object_type'] if d['catalog'] != 'Constellation' else None

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1, 1.3]})
for ax, groups, key, title in [(ax1, cat_groups, lambda d: d['catalog'], 'Documents used as a gold answer, by catalog'),
                               (ax2, type_groups, type_key, 'Documents used as a gold answer, by object type')]:
    base = coverage(gold_base, key, groups)
    full = coverage(gold_all, key, groups)
    y = np.arange(len(groups))[::-1]
    ax.barh(y + 0.19, [base[g] for g in groups], height=0.36, color=SKY, label=f'golden.jsonl ({len(golden)} questions)')
    ax.barh(y - 0.19, [full[g] for g in groups], height=0.36, color=PINK, label=f'+ golden_charles.jsonl ({len(golden) + len(golden_charles)} questions)')
    ax.set_yticks(y); ax.set_yticklabels(groups)
    ax.set_xlabel('% of the documents of that group'); ax.set_title(title)
    ax.set_xlim(0, max(max(full.values()), 1) * 1.15)
ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), fontsize=9)
plt.tight_layout(); plt.show()

n_obj = len(objects)
def obj_ids(s): return [x for x in s if doc_by_id[x]['catalog'] != 'Constellation']
def south(s): return sum(1 for x in obj_ids(s) if doc_by_id[x]['metadata']['dec_deg'] is not None and doc_by_id[x]['metadata']['dec_deg'] < 48.85 - 70)

print(f'Questions: {len(golden)} (golden) + {len(golden_charles)} (ours); questions without gold documents (traps): {n_trap}')
print(f'Retrieval check, project definition: {hits}/{n_gold} gold documents retrieved -> missed: {missed}\n')
print(f"{'':32s} {'golden':>8s} {'+ ours':>8s}")
print(f"{'distinct gold documents':32s} {len(gold_base):8d} {len(gold_all):8d}")
print(f"{'  deep-sky objects (of ' + str(n_obj) + ')':32s} {len(obj_ids(gold_base)):8d} {len(obj_ids(gold_all)):8d}")
print(f"{'  share of all deep-sky objects':32s} {len(obj_ids(gold_base)) / n_obj:8.1%} {len(obj_ids(gold_all)) / n_obj:8.1%}")
print(f"{'  Herschel objects (of 200)':32s} {sum(doc_by_id[x]['catalog'] == 'Herschel' for x in gold_base):8d} {sum(doc_by_id[x]['catalog'] == 'Herschel' for x in gold_all):8d}")
print(f"{'  objects too low from Paris':32s} {south(gold_base):8d} {south(gold_all):8d}")
print(f"{'  objects without magnitude':32s} {sum(doc_by_id[x]['metadata']['magnitude'] is None for x in obj_ids(gold_base)):8d} {sum(doc_by_id[x]['metadata']['magnitude'] is None for x in obj_ids(gold_all)):8d}")

**Takeaway.** The 40 original questions rely on only 33 distinct gold documents: 21 deep-sky objects (5% of the 418) and 12 constellations. **No Herschel object is tested at all**, although Herschel makes up almost half of the objects, and only one of the 21 objects is too low to be observed from Paris. The reported recall of 0.99 (66 of 67 gold documents found; the only miss is `C020` in `q017`, the dense Cygnus case seen in section 2) is therefore a very good sign, but it is measured on a narrow slice of the corpus and should not be read as a guarantee for the other 95% of objects. Our 19 questions broaden the slice (2 Herschel objects, southern objects, an object without magnitude, more object types and traps), but they remain a small sample: they have not been run through the agent yet, and a larger test set would be the natural next step.

## Summary

| Question | Main finding |
|---|---|
| Object types | 48% open clusters (mostly Herschel); rare types have only 3-4 documents |
| Constellations | Milky Way constellations hold 19-23 objects; 20 constellations have none |
| Magnitudes | Median about 7.4; Caldwell is the faintest (56% beyond binoculars); Herschel stops at 8.5 |
| Missing data | 27 objects without magnitude and 3 without coordinates, all coming from OpenNGC |
| Text sizes | Chunks cap at ~234 words (median 155); documents range from 1 to 49 chunks (median 4) |
| Observability | Only 64% of objects are usable from Paris (88 never rise), so visibility must be computed |
| Test coverage | The 40 original questions test 21 of 418 objects and no Herschel object; our 19 questions widen this |